# Case-Control WGS Multi-Ancestry 

## Setup, imports and configuration

In [ ]:
import os, csv, gzip, subprocess, tempfile, pathlib
import pandas as pd
import numpy as np
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime

# R12 ancestries
ANCESTRIES    = ["AAC", "AFR", "AJ", "AMR", "CAS", "EAS", "EUR",
                 "FIN", "MDE", "SAS", "CAH"]
LOW_POWER_ANC = {"FIN", "MDE", "SAS"}
INFLATE_WARN  = {"AMR"}
DATASET       = "WGS"

# Tool
PLINK2 = "/home/jupyter/tools/plink2"

# R12 paths
DIR_WSPS  = "/home/jupyter/workspace/ws_files"
DIR_NOVA  = f"{DIR_WSPS}/Novalis_v3_R12"
DIR_RESU  = f"{DIR_NOVA}/Results_union"
HAR_LIST_FILE  = f"{DIR_WSPS}/HARS_files/HARs_merged/HAR_list_phase_1_union.tsv"
PATH_MK   = pathlib.Path.home() / "workspace/gp2_tier2_eu_release12/clinical_data/master_key_release12_final_vwb.csv"

# QC and analysis parameters
MAX_PCS            = 5      # max number of PCs to include as covariates
BASE_COVARS        = ["SEX", "AGE"]
MAC_MIN            = 5      # minimum minor allele count
GENO_MAX           = 0.05   # max per-variant missingness
HWE_P              = 1e-6   # HWE threshold (controls)
USE_FIRTH_FALLBACK = True   # Firth logistic regression when standard regression fails
MAX_WORKERS        = 8      # parallel processes
OVERLAP_THRESHOLD  = 0.50   # minimum fraction of VCF IDs that must be present in the covariate file

# Create results directories
os.makedirs(DIR_RESU, exist_ok=True)
for ANC in ANCESTRIES:
    os.makedirs(f"{DIR_RESU}/CaseControl_{ANC}_{DATASET}", exist_ok=True)

# Load HAR list
with open(HAR_LIST_FILE) as f:
    HARS_ALL = [row[3] for row in csv.reader(f, delimiter='\t')]
print(f"HARs in the union list: {len(HARS_ALL)}")

# Load master key (only needed to rebuild covariates)
if PATH_MK.exists():
    MK = pd.read_csv(str(PATH_MK), low_memory=False)
    print(f"Master key loaded: {len(MK):,} individuals")
else:
    MK = None
    print(f"Master key not found at {PATH_MK}")


# Helpers

def get_vcf_sample_ids(anc, ds=DATASET):
    """Real sample IDs present in the VCFs for one ancestry.

    Reads the first VCF's header in BINARY mode (the #CHROM line).
    Much faster (0.1s) than text mode on DRAGEN VCFs with thousands of '##' lines.
    More reliable than 'bcftools query -l', which isn't always on the kernel's PATH.
    """
    vcf_dir = f"{DIR_WSPS}/results_region_extrac_v3_{ds}/{anc}/InputFiles/Indiv_HARS"
    if not os.path.isdir(vcf_dir):
        return set()
    vcfs = sorted(f for f in os.listdir(vcf_dir)
                  if f.endswith(".vcf.gz") and not f.endswith(".tbi"))
    if not vcfs:
        return set()
    try:
        with gzip.open(os.path.join(vcf_dir, vcfs[0]), 'rb') as fh:
            for line in fh:
                if line.startswith(b'#CHROM'):
                    fields = line.decode().strip().split('\t')
                    # Fixed VCF columns: CHROM POS ID REF ALT QUAL FILTER INFO FORMAT (9)
                    # From index 9 onward: sample IDs
                    return set(fields[9:]) - {''}
                if not line.startswith(b'#'):
                    break  # reached variant rows without finding #CHROM
    except Exception:
        pass
    return set()


def covar_vcf_overlap(covar_path, vcf_ids):
    """Fraction of VCF IDs present in the covariate file's IID column."""
    if not vcf_ids or not os.path.isfile(covar_path):
        return 0.0
    df = pd.read_csv(covar_path, sep=r'\s+', engine='python', usecols=[0, 1])
    iid_col = df.columns[1]
    covar_ids = set(df[iid_col].astype(str))
    return len(covar_ids & vcf_ids) / len(vcf_ids)


def build_covar_fallback(anc, mk_df, vcf_ids, ds=DATASET, max_pcs=MAX_PCS):
    """Rebuild the covariate file from the master key when the existing one doesn't match the VCF.

    Only includes samples with GP2_phenotype_for_qc in {PD, Control}.
    Excludes Prodromal, Other/Unknown and any other category.
    """
    out_path = (f"{DIR_WSPS}/Working_{ds}_v3/{anc}/InputFiles/"
                f"{anc}_covariate_file_wgs_cc.txt")

    if mk_df is None or not vcf_ids:
        return None, 0, 0

    sub = mk_df[
        mk_df['GP2ID'].isin(vcf_ids) &
        mk_df['GP2_phenotype_for_qc'].isin(['PD', 'Control'])
    ].copy()
    if len(sub) == 0:
        return None, 0, 0

    sub['PHENO'] = sub['GP2_phenotype_for_qc'].map({'PD': 2, 'Control': 1})
    sub['SEX']   = sub['biological_sex_for_qc'].map({'Male': 1, 'Female': 2}).fillna(0).astype(int)
    sub['AGE']   = pd.to_numeric(sub['age_at_sample_collection'], errors='coerce')
    sub['FID']   = sub['GP2ID']
    sub['IID']   = sub['GP2ID']

    pc_path = (pathlib.Path.home() /
               f"workspace/gp2_tier2_eu_release12/wgs/dragen_joint_calling/pcs/{anc}_release12.eigenvec")
    pc_cols = []
    if pc_path.exists():
        try:
            pcs = pd.read_csv(str(pc_path), sep=r'\s+', engine='python')
            pcs = pcs.rename(columns={pcs.columns[0]: 'IID'})
            pcs['IID'] = pcs['IID'].astype(str).str.lstrip('#')
            pc_cols = [c for c in pcs.columns if c.upper().startswith('PC')][:max_pcs]
            sub = sub.merge(pcs[['IID'] + pc_cols], on='IID', how='left')
            if sub[pc_cols[0]].notna().mean() < 0.5:
                sub.drop(columns=pc_cols, inplace=True)
                pc_cols = []
        except Exception:
            pc_cols = []

    out_cols = ['FID', 'IID', 'PHENO', 'SEX', 'AGE'] + pc_cols
    out = sub[out_cols].dropna(subset=['PHENO'])
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    out.to_csv(out_path, sep='\t', index=False)
    return out_path, int((out['PHENO'] == 2).sum()), int((out['PHENO'] == 1).sum())


def detect_covar_name(covar_path, base=BASE_COVARS, max_pcs=MAX_PCS):
    """Build the plink2 --covar-name string from the covariate file's header.

    Includes SEX and AGE when present, plus up to MAX_PCS=5 PCs.
    Caps at 5 even if the file has 10 (the covariate builder's format).
    """
    if not os.path.isfile(covar_path):
        return None, 0
    with open(covar_path) as fh:
        header = fh.readline().strip().replace('#', '').split()
    cols = set(header)
    present_base = [c for c in base if c in cols]
    pcs = [f"PC{i}" for i in range(1, max_pcs + 1) if f"PC{i}" in cols]
    if not present_base:
        return None, 0
    return ",".join(present_base + pcs), len(pcs)


def normalize_covar(covar_path, covar_name=None):
    """Write a clean temp file for plink2 with FID=IID and only the needed columns.

    The covariate builder produces files with FATID/MATID columns (always 0 in
    GP2, unrelated samples) and up to 10 PCs. When plink2 gets a --pheno file
    with those extra columns, it can interpret PHENO as quantitative and run a
    LINEAR regression instead of logistic, producing .glm.linear instead of
    .glm.logistic.hybrid (the pipeline counts those as FAIL).

    This helper:
      1. Sets FID = IID (needed since VCF->PGEN conversion uses --double-id)
      2. Drops FATID, MATID and extra PCs - keeps only FID/IID/PHENO + covariates
      3. Ensures PHENO is an integer 1/2 so plink2 detects it as binary
    """
    df = pd.read_csv(covar_path, sep=r'\s+', engine='python')
    cols = list(df.columns)
    iid_cands = [c for c in cols if c.upper().lstrip('#') == 'IID']
    iid_col   = iid_cands[0] if iid_cands else cols[1]
    df[cols[0]] = df[iid_col]  # FID <- IID

    # Keep only the needed columns (drops FATID, MATID, PC6-PC10, etc.)
    needed = [cols[0], iid_col, 'PHENO']
    if covar_name:
        needed += [c.strip() for c in covar_name.split(',')]
    needed = list(dict.fromkeys(c for c in needed if c in df.columns))
    df = df[needed]

    # PHENO as integer for binary detection (1=Control, 2=PD)
    if 'PHENO' in df.columns:
        df['PHENO'] = pd.to_numeric(df['PHENO'], errors='coerce').fillna(-9).astype(int)

    tmp = tempfile.NamedTemporaryFile(suffix='_covar_norm.txt', delete=False, mode='w')
    df.to_csv(tmp.name, sep='\t', index=False, na_rep='-9')  # NaN as -9 (plink2 convention)
    tmp.close()
    return tmp.name


# Pick the best covariate file per ancestry
COVAR_NAME_BY_ANC = {}
COVAR_PATH_BY_ANC = {}

print(f"\n{'ANC':<6} {'FLAG':<12} {'VCFs':>6} {'PD':>6} {'CTRL':>6} {'#PC':>4}  {'COVAR_SRC':<20}  COVAR_NAME")
print("-" * 105)

for ANC in ANCESTRIES:
    vcf_dir  = f"{DIR_WSPS}/results_region_extrac_v3_{DATASET}/{ANC}/InputFiles/Indiv_HARS"
    existing = f"{DIR_WSPS}/Working_{DATASET}_v3/{ANC}/InputFiles/{ANC}_covariate_file.txt"

    flag = ""
    if ANC in LOW_POWER_ANC: flag = "LOW_POW"
    elif ANC in INFLATE_WARN: flag = "lam=1.24"

    try:
        files = set(os.listdir(vcf_dir))
        n_vcf = sum(1 for h in HARS_ALL
                    if f"{h}.vcf.gz" in files and f"{h}.vcf.gz.tbi" in files)
    except FileNotFoundError:
        n_vcf = 0

    vcf_ids = get_vcf_sample_ids(ANC)
    overlap = covar_vcf_overlap(existing, vcf_ids)

    if overlap >= OVERLAP_THRESHOLD:
        covar_path = existing
        src = "existing"
        df_c = pd.read_csv(covar_path, sep=r'\s+', engine='python')
        iid_col   = next(c for c in df_c.columns if c.upper().lstrip('#') == 'IID')
        pheno_col = next((c for c in df_c.columns if c.upper() == 'PHENO'), None)
        mask = df_c[iid_col].astype(str).isin(vcf_ids)
        if pheno_col:
            n_pd   = int((df_c.loc[mask, pheno_col] == 2).sum())
            n_ctrl = int((df_c.loc[mask, pheno_col] == 1).sum())
        else:
            n_pd = n_ctrl = -1
    else:
        covar_path, n_pd, n_ctrl = build_covar_fallback(ANC, MK, vcf_ids)
        src = f"rebuilt(ov={overlap:.0%})"

    cname, npc = detect_covar_name(covar_path) if covar_path else (None, 0)
    COVAR_NAME_BY_ANC[ANC] = cname
    COVAR_PATH_BY_ANC[ANC] = covar_path

    pd_str   = str(n_pd)   if n_pd   >= 0 else "?"
    ctrl_str = str(n_ctrl) if n_ctrl >= 0 else "?"
    print(f"{ANC:<6} {flag:<12} {n_vcf:>6} {pd_str:>6} {ctrl_str:>6} {npc:>4}  {src:<20}  {cname if cname else '--'}")

## PLINK2 GLM with Firth fallback

In [ ]:
import glob

def _result_files(glm_prefix):
    return [Path(f"{glm_prefix}.PHENO.glm.logistic.hybrid"),
            Path(f"{glm_prefix}.PHENO.glm.firth.hybrid")]

def _has_result(glm_prefix):
    return any(p.exists() and p.stat().st_size > 0 for p in _result_files(glm_prefix))


def cc_one_har(args):
    har, plink2, vcf_dir, covar_norm, out_dir, covar_name = args
    vcf    = str(Path(vcf_dir) / f"{har}.vcf.gz")
    prefix = str(Path(out_dir) / f"{har}")
    glm_p  = prefix + "_glm"

    if _has_result(glm_p):
        return f"[CACHED] {har}"
    if not Path(vcf).exists():
        return f"[SKIP_VCF] {har}"

    # Step 1: VCF -> PGEN
    r1 = subprocess.run(
        [plink2, "--vcf", vcf, "--make-pgen", "--double-id",
         "--out", prefix, "--max-alleles", "2"],
        capture_output=True, text=True
    )
    if not Path(f"{prefix}.pgen").exists():
        return f"[SKIP_PGEN] {har} | {r1.stderr[-150:].strip()}"

    # Step 2: GLM with Firth fallback + QC filters
    glm_mode = ["firth-fallback"] if USE_FIRTH_FALLBACK else []
    cmd = [
        plink2, "--pfile", prefix,
        "--glm", *glm_mode, "hide-covar", "cols=+orbeta,+ci",
        "--pheno",      covar_norm, "--pheno-name", "PHENO",
        "--covar",      covar_norm, "--covar-name", covar_name,
        "--covar-variance-standardize",
        "--mac",  str(MAC_MIN),
        "--geno", str(GENO_MAX),
        "--hwe",  str(HWE_P), "keep-fewhet",
        "--ci",   "0.95",
        "--out",  glm_p,
    ]
    r2 = subprocess.run(cmd, capture_output=True, text=True)

    if _has_result(glm_p):
        firth = Path(f"{glm_p}.PHENO.glm.firth.hybrid")
        tag   = "DONE_FIRTH" if (firth.exists() and firth.stat().st_size > 0) else "DONE"
        return f"[{tag}] {har}"

    # plink2 sometimes writes errors to stdout instead of stderr
    err = (r2.stdout + r2.stderr)[-200:].strip()
    return f"[FAIL] {har} | {err}"


# Run per ancestry
ts_global = datetime.now()

for ANC in ANCESTRIES:
    covar_name = COVAR_NAME_BY_ANC.get(ANC)
    covar_orig = COVAR_PATH_BY_ANC.get(ANC)

    if not covar_name or not covar_orig:
        print(f"\n[SKIP] {ANC}: covariate file missing or has no covariates")
        continue

    vcf_dir = f"{DIR_WSPS}/results_region_extrac_v3_{DATASET}/{ANC}/InputFiles/Indiv_HARS"
    out_dir = f"{DIR_RESU}/CaseControl_{ANC}_{DATASET}"

    try:
        files = set(os.listdir(vcf_dir))
    except FileNotFoundError:
        print(f"\n[SKIP] {ANC}: VCF directory not found")
        continue

    present = [h for h in HARS_ALL
               if f"{h}.vcf.gz" in files and f"{h}.vcf.gz.tbi" in files]
    if not present:
        print(f"\n[SKIP] {ANC}: 0 HARs with a VCF")
        continue

    # Clean up stale .glm.linear files (incorrect linear regression on a binary PHENO,
    # caused by FATID/MATID columns in the covariate file - fixed by normalize_covar)
    stale_linear = glob.glob(f"{out_dir}/*_glm.PHENO.glm.linear")
    if stale_linear:
        print(f"  Removing {len(stale_linear)} stale .glm.linear files (incorrect regression)")
        for f in stale_linear:
            try: os.unlink(f)
            except: pass

    # Normalize covariate: FID=IID, strip extra columns, PHENO as integer
    # Pass covar_name so normalize_covar knows which columns to keep
    covar_norm = normalize_covar(covar_orig, covar_name)

    low_pow_tag = " [LOW_POWER]"  if ANC in LOW_POWER_ANC else ""
    infl_tag    = " [lam_GC=1.24]" if ANC in INFLATE_WARN  else ""
    print(f"\n=== {ANC} - {len(present)} HARs | {covar_name} | Workers: {MAX_WORKERS}{low_pow_tag}{infl_tag} ===")
    ts_anc = datetime.now()

    tasks = [(h, PLINK2, vcf_dir, covar_norm, out_dir, covar_name) for h in present]
    n_done = n_firth = n_cached = n_skip = n_fail = 0
    recent_fails = []

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(cc_one_har, t): t for t in tasks}
        for i, fut in enumerate(as_completed(futures), 1):
            msg = fut.result()
            if   msg.startswith("[DONE_FIRTH"): n_done += 1; n_firth += 1
            elif msg.startswith("[DONE"):       n_done += 1
            elif msg.startswith("[CACHED"):     n_cached += 1
            elif msg.startswith("[SKIP"):       n_skip += 1
            else:
                n_fail += 1
                recent_fails.append(msg)

            if i % 200 == 0:
                elapsed = (datetime.now() - ts_anc).total_seconds()
                rate    = i / max(elapsed / 60, 0.01)
                eta     = (len(tasks) - i) / max(rate, 0.01)
                print(f"  [{i}/{len(tasks)}] D={n_done}(F={n_firth}) C={n_cached} "
                      f"FAIL={n_fail} SKIP={n_skip} | {rate:.0f}/min | ETA {eta:.0f}min")
                if recent_fails:
                    for fm in recent_fails[-3:]:
                        print(f"    {fm}")
                    recent_fails.clear()

    try:
        os.unlink(covar_norm)
    except Exception:
        pass

    print(f"  {ANC} done in {datetime.now()-ts_anc} - "
          f"DONE={n_done} (Firth={n_firth}) CACHED={n_cached} FAIL={n_fail} SKIP={n_skip}")
    if recent_fails:
        for fm in recent_fails:
            print(f"    {fm}")

print(f"\nTotal time: {datetime.now() - ts_global}")

## Compile results and compute FDR-BH


In [ ]:
import os, csv
import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.stats.multitest import multipletests

ANCESTRIES    = ["AAC", "AFR", "AJ", "AMR", "CAS", "EAS", "EUR",
                 "FIN", "MDE", "SAS", "CAH"]
LOW_POWER_ANC = {"FIN", "MDE", "SAS"}
INFLATE_WARN  = {"AMR"}
DATASET       = "WGS"
DIR_WSPS      = "/home/jupyter/workspace/ws_files"
DIR_NOVA      = f"{DIR_WSPS}/Novalis_v3_R12"
DIR_RESU      = f"{DIR_NOVA}/Results_union"
HAR_LIST_FILE      = f"{DIR_WSPS}/HARS_files/HARs_merged/HAR_list_phase_1_union.tsv"
os.makedirs(DIR_RESU, exist_ok=True)

with open(HAR_LIST_FILE) as f:
    HARS_ALL = [row[3] for row in csv.reader(f, delimiter='\t')]
BONF_PER_ANC = 0.05 / len(HARS_ALL)


def _read_glm(path, model_tag):
    """Read a plink2 .glm.{logistic,firth}.hybrid file and keep only TEST=ADD rows."""
    df = pd.read_csv(path, sep='\t')
    if "TEST" in df.columns:
        df = df[df["TEST"] == "ADD"].copy()
    if df.empty:
        return None
    df.rename(columns={'LOG(OR)_SE': 'SE'}, errors='ignore', inplace=True)
    df["MODEL"] = model_tag
    return df


all_dfs = []
for ANC in ANCESTRIES:
    out_dir = Path(f"{DIR_RESU}/CaseControl_{ANC}_{DATASET}")
    if not out_dir.is_dir():
        print(f"[SKIP] {ANC}: directory doesn't exist")
        continue

    rows, n_err = [], 0
    for suffix, tag in [("logistic", "logistic"), ("firth", "firth")]:
        for f in out_dir.glob(f"*_glm.PHENO.glm.{suffix}.hybrid"):
            if f.stat().st_size == 0:
                continue
            har = f.name.split("_glm.PHENO")[0]
            try:
                d = _read_glm(f, tag)
                if d is not None:
                    d["HAR"]      = har
                    d["ANCESTRY"] = ANC
                    d["DATASET"]  = DATASET
                    rows.append(d)
            except Exception:
                n_err += 1

    if not rows:
        print(f"[SKIP] {ANC}: 0 readable files (errors={n_err})")
        continue

    df_anc = pd.concat(rows, ignore_index=True)
    df_anc["P"] = pd.to_numeric(df_anc["P"], errors="coerce")
    mask = df_anc["P"].notna() & (df_anc["P"] > 0)
    df_anc["FDR_BH"]           = np.nan
    df_anc["Bonferroni_thresh"] = BONF_PER_ANC
    df_anc["LOW_POWER"]         = ANC in LOW_POWER_ANC
    df_anc["LAMBDA_WARN"]       = ANC in INFLATE_WARN

    if mask.sum() > 0:
        _, q, _, _ = multipletests(df_anc.loc[mask, "P"].astype(float), method="fdr_bh")
        df_anc.loc[mask, "FDR_BH"] = q

    out_tsv = f"{DIR_RESU}/CaseControl_{ANC}_{DATASET}_FDRv3.tsv"
    df_anc.to_csv(out_tsv, sep='\t', index=False)

    n_har  = df_anc["HAR"].nunique()
    n_bonf = int((df_anc["P"] < BONF_PER_ANC).sum())
    n_fdr  = int((df_anc["FDR_BH"] < 0.05).sum())
    flag   = " [LOW_POWER]" if ANC in LOW_POWER_ANC else (" [lam=1.24]" if ANC in INFLATE_WARN else "")
    print(f"{ANC:<6}{flag:<13} {len(df_anc):>9,} var | {n_har:>5} HARs | Bonf={n_bonf} | FDR<0.05={n_fdr}")
    all_dfs.append(df_anc)

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    df_all.to_csv(f"{DIR_RESU}/ALL_CC_WGS_FDRv3.tsv", sep='\t', index=False)
    print(f"\nGlobal: {DIR_RESU}/ALL_CC_WGS_FDRv3.tsv | {len(df_all):,} variants")
    df_prim = df_all[~df_all["LOW_POWER"]]
    print(f"Primary analysis (excluding LOW_POWER): {len(df_prim):,} variants")
    print(f"Primary Bonferroni hits (p<{BONF_PER_ANC:.2e}): {(df_prim['P'] < BONF_PER_ANC).sum()}")
    print(f"Primary FDR-BH < 0.05: {(df_prim['FDR_BH'] < 0.05).sum()}")

## Diagnostics - P-value distribution and lambda_GC per ancestry


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from scipy.stats import chi2

ANCESTRIES    = ["AAC", "AFR", "AJ", "AMR", "CAS", "EAS", "EUR",
                 "FIN", "MDE", "SAS", "CAH"]
LOW_POWER_ANC = {"FIN", "MDE", "SAS"}
DATASET       = "WGS"
DIR_RESU      = "/home/jupyter/workspace/ws_files/Novalis_v3_R12/Results_union"

def lambda_gc(p):
    p = p[(p > 0) & (p <= 1)]
    if len(p) == 0:
        return np.nan
    obs = chi2.isf(p, df=1)
    return np.median(obs) / 0.4549

print(f"{'ANC':<6} {'FLAG':<9} {'nVar':>8} {'minP':>11} {'P<1e-4':>8} {'P<1e-5':>8} {'lam_GC':>8}")
print("-" * 65)
frames = []
for ANC in ANCESTRIES:
    tsv = Path(f"{DIR_RESU}/CaseControl_{ANC}_{DATASET}_FDRv3.tsv")
    if not tsv.exists():
        print(f"{ANC:<6} (no TSV)"); continue
    df = pd.read_csv(tsv, sep='\t')
    df["P"] = pd.to_numeric(df["P"], errors="coerce")
    p   = df["P"].dropna().values
    lam = lambda_gc(p)
    flag = "LOW_POW" if ANC in LOW_POWER_ANC else ""
    lam_warn = " !!" if lam > 1.1 else ""
    frames.append(df)
    print(f"{ANC:<6} {flag:<9} {len(p):>8,} {p.min():>11.2e} "
          f"{(p<1e-4).sum():>8} {(p<1e-5).sum():>8} {lam:>7.3f}{lam_warn}")

if frames:
    allc = pd.concat(frames, ignore_index=True)
    allc["P"] = pd.to_numeric(allc["P"], errors="coerce")
    prim = allc[~allc["LOW_POWER"].fillna(False)].copy()

    cols = [c for c in ['ANCESTRY', 'HAR', '#CHROM', 'POS', 'ID', 'A1',
                         'A1_FREQ', 'OR', 'SE', 'L95', 'U95', 'P', 'FDR_BH',
                         'MODEL', 'LAMBDA_WARN'] if c in prim.columns]
    print(f"\nTop 20 variants - primary analysis (excluding FIN/MDE/SAS):")
    print(prim.sort_values('P').head(20)[cols].to_string(index=False))